# Intermediate: Advanced Training Techniques

Master advanced techniques to improve model training and performance.

Topics covered:
- Learning rate scheduling
- Gradient clipping
- Batch normalization
- Dropout and regularization
- Early stopping
- Mixed precision training
- Gradient accumulation

## Why Advanced Techniques?

Simple training often leads to:
- Slow convergence
- Poor generalization
- Training instability
- Suboptimal performance

Advanced techniques solve these issues!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


## 1. Learning Rate Scheduling

Adjust learning rate during training for better convergence.

```
LR Schedule Types:

StepLR:      ▬▬▬▬▬▬▬▬▬___▬▬▬▬▬___▬▬▬
ExponentialLR: ▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬▬
CosineAnnealing: ▬▬▬▬▬∼∼∼∼▬▬▬▬▬∼∼∼∼
ReduceLROnPlateau: ▬▬▬▬▬___▬▬▬▬▬_____
```


In [ ]:
# Setup dummy model and optimizer
model = nn.Linear(10, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)

# 1. StepLR - reduce LR every N epochs
scheduler_step = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# 2. Exponential LR - multiply LR by gamma each epoch
scheduler_exp = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

# 3. Cosine Annealing - cosine decay
scheduler_cosine = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

# 4. Reduce on Plateau - reduce when metric plateaus
scheduler_plateau = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', 
                                                         patience=5, factor=0.1)

# 5. One Cycle - popular for fast training
scheduler_onecycle = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.1, 
                                                   steps_per_epoch=100, epochs=1)

print("Schedulers created!")

# Visualize schedules
lrs = []
for epoch in range(100):
    lrs.append(optimizer.param_groups[0]['lr'])
    optimizer.step()
    scheduler_cosine.step()

plt.figure(figsize=(10, 4))
plt.plot(lrs)
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Cosine Annealing Schedule')
plt.grid(True)
plt.show()


## 2. Gradient Clipping

Prevent exploding gradients by limiting gradient magnitude.

```
Without Clipping:          With Clipping:
grad = 1000 →→→→→→→→→      grad = 10 →→→
(unstable)                 (stable)
```


In [ ]:
def train_with_clipping(model, loader, optimizer, criterion, clip_value=1.0):
    model.train()
    total_loss = 0
    
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        
        # Method 1: Clip by value
        torch.nn.utils.clip_grad_value_(model.parameters(), clip_value)
        
        # Method 2: Clip by norm (more common)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_value)
        
        optimizer.step()
        total_loss += loss.item()
    
    return total_loss / len(loader)

# Check gradient norms
def check_gradients(model):
    total_norm = 0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    total_norm = total_norm ** 0.5
    return total_norm

print("Gradient clipping functions defined!")


## 3. Batch Normalization

Normalize activations for faster and more stable training.

```
Without BatchNorm:        With BatchNorm:
Layer 1: [0.1, 0.5, ...]  Layer 1: [0.0, 0.0, ...]
Layer 2: [10, 50, ...]    Layer 2: [0.0, 0.0, ...]
Layer 3: [1000, ...]      Layer 3: [0.0, 0.0, ...]
(internal covariate       (normalized)
 shift)
```


In [ ]:
class CNNWithBatchNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)  # After conv, before activation
        
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        
        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.bn3 = nn.BatchNorm1d(256)  # 1D for fully connected
        
        self.fc2 = nn.Linear(256, 10)
        
    def forward(self, x):
        # Conv block 1
        x = self.conv1(x)
        x = self.bn1(x)  # Normalize before activation
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        
        # Conv block 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        
        # Fully connected
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.fc2(x)
        
        return x

model = CNNWithBatchNorm().to(device)
print(model)

# Important: Set model to train/eval mode for BatchNorm
model.train()  # Uses batch statistics
model.eval()   # Uses running statistics

print("\nBatchNorm benefits:")
print("✓ Faster convergence")
print("✓ Higher learning rates possible")
print("✓ Less sensitive to initialization")
print("✓ Regularization effect")


## 4. Dropout and Regularization

Prevent overfitting by randomly dropping neurons during training.

```
Training (p=0.5):      Inference:
[1][X][1][X][1]  →    [0.5][0.5][0.5][0.5][0.5]
(50% dropped)          (scaled outputs)
```


In [ ]:
class ModelWithDropout(nn.Module):
    def __init__(self, dropout_prob=0.5):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.dropout1 = nn.Dropout(dropout_prob)
        
        self.fc2 = nn.Linear(512, 256)
        self.dropout2 = nn.Dropout(dropout_prob)
        
        self.fc3 = nn.Linear(256, 10)
        
    def forward(self, x):
        x = x.view(-1, 784)
        
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)  # Apply dropout
        
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        
        x = self.fc3(x)  # No dropout on output layer
        return x

# L2 Regularization (Weight Decay)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# L1 Regularization (manual)
def l1_regularization(model, lambda_l1=0.01):
    l1_loss = 0
    for param in model.parameters():
        l1_loss += torch.sum(torch.abs(param))
    return lambda_l1 * l1_loss

print("Regularization techniques demonstrated!")


## 5. Early Stopping

Stop training when validation performance stops improving.

```
Val Loss:
  |
  |  ●
  |    ●
  |      ●  ← best
  |        ○ ○ ○  ← stop here
  |_________________ Epochs
     (patience=3)
```


In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model = None
        
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0
    
    def save_checkpoint(self, model):
        if self.verbose:
            print(f'Validation loss decreased ({self.best_loss:.6f}). Saving model...')
        self.best_model = model.state_dict().copy()

# Usage
early_stopping = EarlyStopping(patience=10, verbose=True)

for epoch in range(100):
    train_loss = 0  # train_one_epoch(model, train_loader)
    val_loss = 0    # validate(model, val_loader)
    
    early_stopping(val_loss, model)
    
    if early_stopping.early_stop:
        print("Early stopping triggered!")
        break

# Load best model
# model.load_state_dict(early_stopping.best_model)

print("Early stopping class created!")


## 6. Mixed Precision Training

Use FP16 (half precision) to speed up training and reduce memory.

```
FP32 (Float32):     FP16 (Float16):
32 bits             16 bits
More memory         Less memory
Slower              Faster (on modern GPUs)
More accurate       Slightly less accurate
```


In [ ]:
# Mixed precision training (requires CUDA)
from torch.cuda.amp import autocast, GradScaler

def train_mixed_precision(model, loader, optimizer, criterion):
    model.train()
    scaler = GradScaler()  # Gradient scaler for FP16
    
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass in FP16
        with autocast():
            output = model(data)
            loss = criterion(output, target)
        
        # Backward pass with scaled gradients
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    
    return loss.item()

# Benefits:
# - 2-3x faster training (on modern GPUs)
# - 50% less memory usage
# - Minimal accuracy loss

print("Mixed precision training demonstrated!")
print("Note: Requires CUDA-capable GPU")


## 7. Gradient Accumulation

Simulate larger batch sizes with limited memory.

```
Regular (batch=64):
  Forward → Backward → Step

Accumulated (simulated batch=256):
  Forward → Backward → 
  Forward → Backward → 
  Forward → Backward → 
  Forward → Backward → Step
  (4 steps, each batch=64)
```


In [ ]:
def train_with_accumulation(model, loader, optimizer, criterion, accumulation_steps=4):
    model.train()
    optimizer.zero_grad()
    
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        
        output = model(data)
        loss = criterion(output, target)
        
        # Normalize loss by accumulation steps
        loss = loss / accumulation_steps
        loss.backward()
        
        # Update weights every N steps
        if (batch_idx + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
    
    return loss.item()

print("Gradient accumulation demonstrated!")
print("Effective batch size = batch_size × accumulation_steps")


## 8. Complete Training Pipeline with All Techniques


In [ ]:
class AdvancedTrainer:
    def __init__(self, model, train_loader, val_loader, device):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        
        # Optimizer with weight decay
        self.optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        
        # Learning rate scheduler
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=100, eta_min=1e-6
        )
        
        # Loss and scaler
        self.criterion = nn.CrossEntropyLoss()
        self.scaler = GradScaler()
        
        # Early stopping
        self.early_stopping = EarlyStopping(patience=10)
        
        # History
        self.history = {'train_loss': [], 'val_loss': [], 'lr': []}
    
    def train_epoch(self):
        self.model.train()
        total_loss = 0
        
        for data, target in self.train_loader:
            data, target = data.to(self.device), target.to(self.device)
            
            self.optimizer.zero_grad()
            
            with autocast():
                output = self.model(data)
                loss = self.criterion(output, target)
            
            self.scaler.scale(loss).backward()
            
            # Gradient clipping
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            self.scaler.step(self.optimizer)
            self.scaler.update()
            
            total_loss += loss.item()
        
        return total_loss / len(self.train_loader)
    
    def validate(self):
        self.model.eval()
        total_loss = 0
        
        with torch.no_grad():
            for data, target in self.val_loader:
                data, target = data.to(self.device), target.to(self.device)
                output = self.model(data)
                loss = self.criterion(output, target)
                total_loss += loss.item()
        
        return total_loss / len(self.val_loader)
    
    def train(self, epochs):
        for epoch in range(epochs):
            train_loss = self.train_epoch()
            val_loss = self.validate()
            
            # Update scheduler
            self.scheduler.step()
            
            # Save history
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['lr'].append(self.optimizer.param_groups[0]['lr'])
            
            # Early stopping
            self.early_stopping(val_loss, self.model)
            if self.early_stopping.early_stop:
                print(f"Early stopping at epoch {epoch}")
                break
            
            if epoch % 10 == 0:
                print(f"Epoch {epoch}: Train Loss: {train_loss:.4f}, "
                      f"Val Loss: {val_loss:.4f}, LR: {self.history['lr'][-1]:.6f}")
        
        # Load best model
        self.model.load_state_dict(self.early_stopping.best_model)
        return self.history

print("Advanced trainer class created!")
print("\nFeatures included:")
print("✓ Mixed precision training")
print("✓ Gradient clipping")
print("✓ Learning rate scheduling")
print("✓ Early stopping")
print("✓ Weight decay")
print("✓ Training history tracking")


## 📝 Summary

✅ Implemented learning rate scheduling  
✅ Applied gradient clipping  
✅ Used batch normalization  
✅ Added dropout and regularization  
✅ Implemented early stopping  
✅ Enabled mixed precision training  
✅ Used gradient accumulation  
✅ Built complete advanced training pipeline  

### Best Practices

1. **Always use learning rate scheduling**
2. **Clip gradients for RNNs**
3. **Use BatchNorm in CNNs**
4. **Add Dropout to prevent overfitting**
5. **Implement early stopping**
6. **Use mixed precision on modern GPUs**
7. **Monitor training with TensorBoard**

### Next Steps

- **Next**: `04_transfer_learning.ipynb`
- **Practice**: Apply these techniques to your models
- **Challenge**: Achieve 99%+ accuracy on MNIST


## 🎯 Practice Exercises


In [ ]:
# Exercise 1: Compare different LR schedules
# Your code here:


# Exercise 2: Train with and without BatchNorm, compare results
# Your code here:


# Exercise 3: Implement custom learning rate warmup
# Your code here:


# Exercise 4: Build a complete trainer with logging to TensorBoard
# Your code here:
